Sascha Spors,
Professorship Signal Theory and Digital Signal Processing,
Institute of Communications Engineering (INT),
Faculty of Computer Science and Electrical Engineering (IEF),
University of Rostock,
Germany

# Data Driven Audio Signal Processing - A Tutorial with Computational Examples

Winter Semester 2025/26 (Master Course #24512)

- lecture: https://github.com/spatialaudio/data-driven-audio-signal-processing-lecture
- tutorial: https://github.com/spatialaudio/data-driven-audio-signal-processing-exercise

Feel free to contact lecturer frank.schultz@uni-rostock.de

# Binary logistic regression model with hidden layers and a sigmoid output layer

- we use **PyTorch** to train the model and to make predictions
- we use scikit-learn for data synthesis and split
- we use scikit-learn for statistical measures
- see [binary_logistic_regression_tf_with_hidden_layers.ipynb](binary_logistic_regression_tf_with_hidden_layers.ipynb) for a TensorFlow implementation of the same problem

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sklearn
import torch

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
from sklearn.metrics import accuracy_score, balanced_accuracy_score

from torch.utils.data import TensorDataset, DataLoader
from torchinfo import summary

torch.__version__, sklearn.__version__  # last manual check with 2.8.0, 1.7.2

## Synthesis of Data

In [ ]:
M = int(5 / 4 * 80000)  # number of samples per feature
N = 2  # number of features

train_size = 4 / 5  # 80% of data are used for training

# these seeds produce 'nice' two classes each with
# two clusters for chosen M, N and train_size
random_state_idx = 0
random_state = np.array([7, 21, 24, 25, 29, 33, 38])
X, Y = make_classification(
    n_samples=M,
    n_features=N,
    n_informative=N,
    n_redundant=0,
    n_classes=2,
    n_clusters_per_class=2,
    class_sep=1.5,
    flip_y=1e-2,
    random_state=random_state[random_state_idx],
)
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, train_size=train_size, random_state=None
)

## Plot Data

In [ ]:
plt.figure(figsize=(10, 5))
plt.subplot(1, 3, 1)
plt.plot(X[Y == 1, 0],
         X[Y == 1, 1],
         "o", color='orangered', ms=1)
plt.plot(X[Y == 0, 0],
         X[Y == 0, 1],
         "o", color='dodgerblue', ms=1)
plt.axis("square")
plt.title("data 1 and 0 " + str(X.shape))
plt.xlabel("feature 1")
plt.ylabel("feature 2")
plt.axis([-10, 10, -10, 10])
plt.grid()

plt.subplot(1, 3, 2)
plt.plot(X[Y == 1, 0],
         X[Y == 1, 1],
         "o", color='orangered', ms=1)
plt.axis("square")
plt.title("data 1 " + str(X.shape))
plt.xlabel("feature 1")
plt.ylabel("feature 2")
plt.axis([-10, 10, -10, 10])
plt.grid()

plt.subplot(1, 3, 3)
plt.plot(X[Y == 0, 0],
         X[Y == 0, 1],
         "o", color='dodgerblue', ms=1)
plt.axis("square")
plt.title("data 0 " + str(X.shape))
plt.xlabel("feature 1")
plt.ylabel("feature 2")
plt.axis([-10, 10, -10, 10])
plt.grid()

## Learning Parameters

In [ ]:
batch_size = 2**3
num_epochs = 5
learning_rate = 0.01

## Prepare Data for Torch & Scikit-Learn

In [ ]:
Y_train = Y_train[:, np.newaxis]
Y_test = Y_test[:, np.newaxis]

data_train = TensorDataset(torch.FloatTensor(X_train),
                           torch.FloatTensor(Y_train))
data_train_loader = DataLoader(dataset=data_train,
                               batch_size=batch_size,
                               shuffle=True)

## Define Torch Model

In [ ]:
class Model(torch.nn.Module):

    def __init__(self, input_size):
        super(Model, self).__init__()

        self.linear1 = torch.nn.Linear(input_size, 2)
        self.act1 = torch.nn.Tanh()
        self.linear2 = torch.nn.Linear(2, 2)
        self.act2 = torch.nn.Tanh()
        self.linear3 = torch.nn.Linear(2, 1)
        self.sigmoid = torch.nn.Sigmoid()

    def forward(self, x):
        x = self.linear1(x)
        x = self.act1(x)
        x = self.linear2(x)
        x = self.act2(x)
        x = self.linear3(x)
        x = self.sigmoid(x)
        return x
    
    def predict_class(self, x):
        pred = self.forward(x)
        return (pred >= 0.5).float()
    
model = Model(input_size=N)
print(summary(model, input_size=(batch_size, N)))
print(next(model.parameters()).device)

## Define Empirical Risk and Optimizer

In [ ]:
empirical_risk = torch.nn.BCELoss(reduction='mean') 
optimizer = torch.optim.SGD(model.parameters(),
                            lr=learning_rate)

## Put Model onto Hardware

In [ ]:
print('mps available? ', torch.backends.mps.is_available())
#device = torch.device('mps')
device = torch.device('cpu')
model = model.to(device)
print(next(model.parameters()).device)
# check some model weights
model.linear1.weight, model.linear1.bias

## Train the Model

In [ ]:
for epoch in range(num_epochs):
    for i, batch in enumerate(data_train_loader, 1):
        X, Y = batch[0].to(device), batch[1].to(device)
        Y_pred = model(X)  # i.e. model.forward(X), forward prop
        loss = empirical_risk(Y_pred, Y)
        if (i+1) % 1000 == 0:
            print('epoch:', epoch+1, 'batch', i+1)
        loss.backward()  # back prop
        optimizer.step()  # gradient descent
        optimizer.zero_grad()  # reset gradients for next iter
    # all batches per epoch, now do a prediction
    # on train & test so check where we are:
    with torch.no_grad():  # no_grad !!! to not influence the back prop
        er = empirical_risk(model(
            torch.tensor(X_train, dtype=torch.float32).to(device)),
            torch.tensor(Y_train, dtype=torch.float32).to(device))
        print('train loss', er)
        er = empirical_risk(model(
            torch.tensor(X_test, dtype=torch.float32).to(device)),
            torch.tensor(Y_test, dtype=torch.float32).to(device))
        print('test loss', er)
    print('#####\n')
        

## Model Prediction

### Empirical Risk

In [ ]:
with torch.no_grad():
    er = empirical_risk(model.forward(
        torch.tensor(X_train, dtype=torch.float32).to(device)),
        torch.tensor(Y_train, dtype=torch.float32).to(device))
    print('final train loss', er)
    er = empirical_risk(model.forward(
        torch.tensor(X_test, dtype=torch.float32).to(device)),
        torch.tensor(Y_test, dtype=torch.float32).to(device))
    print('final test loss', er)

### Binary Class Metrics 

In [ ]:
with torch.no_grad():

    X_tmp = torch.tensor(X_train, dtype=torch.float32).to(device)
    Y_pred_train = model.predict_class(X_tmp).cpu()

    X_tmp = torch.tensor(X_test, dtype=torch.float32).to(device)
    Y_pred_test = model.predict_class(X_tmp).cpu()

#### Confusion Matrix

In [ ]:
print('train')
print(confusion_matrix(
    y_true=Y_train,
    y_pred=Y_pred_train,
    normalize=None))
print(confusion_matrix(
    y_true=Y_train,
    y_pred=Y_pred_train,
    normalize='all')*100)
print('\n test')
print(confusion_matrix(
    y_true=Y_test,
    y_pred=Y_pred_test,
    normalize=None))
print(confusion_matrix(
    y_true=Y_test,
    y_pred=Y_pred_test,
    normalize='all')*100)

#### Precision, Recall, F1-Score, Support

In [ ]:
p, r, f, s = precision_recall_fscore_support(y_true=Y_train,
                                             y_pred=Y_pred_train)
print('train', p, r, f, s)
p, r, f, s = precision_recall_fscore_support(y_true=Y_test,
                                             y_pred=Y_pred_test)
print('test', p, r, f, s)

#### Accuracy, Balanced Accuracy

We have a very balanced data set, hence both are very close

In [ ]:
a = accuracy_score(y_true=Y_train,
                   y_pred=Y_pred_train)
ba = balanced_accuracy_score(y_true=Y_train,
                             y_pred=Y_pred_train)
print('train:', a, ba)
a = accuracy_score(y_true=Y_test,
                   y_pred=Y_pred_test)
ba = balanced_accuracy_score(y_true=Y_test,
                             y_pred=Y_pred_test)
print('test', a, ba)

## Plot Data Points and Decision Plane

In [ ]:
levels = [0.0, 0.05, 0.45, 0.5, 0.55, 0.95, 1]

f1, f2 = np.arange(-10, 10, 0.1), np.arange(-10, 10, 0.1)
xv, yv = np.meshgrid(f1, f2)
xv_tmp = np.reshape(xv, (xv.shape[0]*xv.shape[1],1))
yv_tmp = np.reshape(yv, (yv.shape[0]*yv.shape[1],1))
X_tmp = torch.tensor(np.hstack([xv_tmp, yv_tmp]),
                     dtype=torch.float32).to(device)
with torch.no_grad():
    Y_tmp = model.predict_class(X_tmp).cpu().detach().numpy()
tmp = np.reshape(Y_tmp, (xv.shape[0],-1))
# hard decision boundary:
#tmp[tmp < 0.5], tmp[tmp >= 0.5] = 0, 1

plt.figure(figsize=(10, 10))
plt.subplot(2, 2, 1)
plt.plot(X_train[Y_train[:, 0] == 1, 0],
         X_train[Y_train[:, 0] == 1, 1],
         "o", color='orangered', ms=1)
plt.contourf(f1, f2, tmp, levels=levels, cmap="RdBu_r")
plt.axis("equal")
plt.colorbar()
plt.title("training " + str(X_train.shape))
plt.xlabel("feature 1")
plt.ylabel("feature 2")

plt.subplot(2, 2, 2)
plt.plot(X_train[Y_train[:, 0] == 0, 0],
         X_train[Y_train[:, 0] == 0, 1],
         "o", color='dodgerblue', ms=1)
plt.contourf(f1, f2, tmp, levels=levels, cmap="RdBu_r")
plt.axis("equal")
plt.colorbar()
plt.title("training " + str(X_train.shape))
plt.xlabel("feature 1")
plt.ylabel("feature 2")

plt.subplot(2, 2, 3)
plt.plot(X_test[Y_test[:, 0] == 1, 0],
         X_test[Y_test[:, 0] == 1, 1],
         "o", color='orangered', ms=1)
plt.contourf(f1, f2, tmp, levels=levels, cmap="RdBu_r")
plt.axis("equal")
plt.colorbar()
plt.title("test " + str(X_test.shape))
plt.xlabel("feature 1")
plt.ylabel("feature 2")

plt.subplot(2, 2, 4)
plt.plot(X_test[Y_test[:, 0] == 0, 0],
         X_test[Y_test[:, 0] == 0, 1],
         "o", color='dodgerblue', ms=1)
plt.contourf(f1, f2, tmp, levels=levels, cmap="RdBu_r")
plt.axis("equal")
plt.colorbar()
plt.title("test " + str(X_test.shape))
plt.xlabel("feature 1")
plt.ylabel("feature 2")

## Copyright

- the notebooks are provided as [Open Educational Resources](https://en.wikipedia.org/wiki/Open_educational_resources)
- feel free to use the notebooks for your own purposes
- the text is licensed under [Creative Commons Attribution 4.0](https://creativecommons.org/licenses/by/4.0/)
- the code of the IPython examples is licensed under the [MIT license](https://opensource.org/licenses/MIT)
- please attribute the work as follows: *Frank Schultz, Data Driven Audio Signal Processing - A Tutorial Featuring Computational Examples, University of Rostock* ideally with relevant file(s), github URL https://github.com/spatialaudio/data-driven-audio-signal-processing-exercise, commit number and/or version tag, year.